<a href="https://colab.research.google.com/github/Ayushsan1/Pytorch-Implementation-and-Foundation/blob/main/Pytorch_JIT_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#PyTorch JIT compiler (Just-in-time) Implementation

In [1]:
import torch
import timeit # to accurately measure the execution time of code blocks, functions, or specific operations
import torch.nn as nn

In [4]:
class Net(nn.Module):
  def __init__(self):
    super(Net, self).__init__()
    self.conv1 = nn.Conv2d(3, 10, 3)

  def forward(self, x):
    return self.conv1(x)

In [20]:
#trying with torch.jit.trace
model = Net()
dummy_forward_input = torch.rand(10, 3, 50, 50)
traced_module = torch.jit.trace(model, dummy_forward_input)

In [21]:
model #regular how it shows

Net(
  (conv1): Conv2d(3, 10, kernel_size=(3, 3), stride=(1, 1))
)

In [22]:
traced_module

Net(
  original_name=Net
  (conv1): Conv2d(original_name=Conv2d)
)

In [23]:
traced_module.graph # this is the IR (intermediate representation) written in C++ which is strongly typed , static computational graph... Represents flow of Pytorch Model

graph(%self.1 : __torch__.___torch_mangle_7.Net,
      %x : Float(10, 3, 50, 50, strides=[7500, 2500, 50, 1], requires_grad=0, device=cpu)):
  %conv1 : __torch__.torch.nn.modules.conv.___torch_mangle_6.Conv2d = prim::GetAttr[name="conv1"](%self.1)
  %33 : Tensor = prim::CallMethod[name="forward"](%conv1, %x)
  return (%33)

In [28]:
 #lets check how performance improves of computation and speed of both wrapped traced module and regular model
a = timeit.timeit()
for i in range(10000):
  model(dummy_forward_input)
b = timeit.timeit()
print(b-a)


-0.00210540799980663


In [29]:
a = timeit.timeit()
for i in range(10000):
  traced_module(dummy_forward_input)
b = timeit.timeit()
print(b-a)

-0.004666324000027089


In [30]:
#Now trying the same with torch.script
model = Net()
dummy_forward_input = torch.rand(10, 3, 50, 50)
script_module = torch.jit.script(model, dummy_forward_input)

/tmp/ipykernel_908/1146518427.py:4: FutureWarning: `optimize` is deprecated and has no effect. Use `with torch.jit.optimized_execution()` instead
  script_module = torch.jit.script(model, dummy_forward_input)


In [31]:
traced_module

RecursiveScriptModule(
  original_name=Net
  (conv1): RecursiveScriptModule(original_name=Conv2d)
)

In [32]:
script_module.graph

graph(%self : __torch__.___torch_mangle_5.Net,
      %x.1 : Tensor):
  %conv1 : __torch__.torch.nn.modules.conv.___torch_mangle_4.Conv2d = prim::GetAttr[name="conv1"](%self)
  %4 : Tensor = prim::CallMethod[name="forward"](%conv1, %x.1) # /tmp/ipykernel_908/1287770099.py:7:11
  return (%4)

We may often/maximum use `torch.jit.script` **as it is because it can handle if-else , different data structures and it has subset of python built into it.**

You check it defining a if else loop then once running it through trace and then through script , and you find how trace leave the if else but script do carry whole structure and if-else too into it.